In [ ]:
"""
GAM Script For Thesis

Created by Anita Taylor
""" 

""" To do
- Match with Aquifer type
- Group by Aquifer type
- drop missing data
- Standardize data
- Run GAM
- Analysis GAM
- Add in previous chart
"""

In [ ]:
import pandas as pd
from pygam import LinearGAM
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
merged_df = pd.merge(df1, df2, left_on='well_number', right_on='OBS_well_number')


In [ ]:
grouped = merged_df.groupby('aquifer_type')

# A dictionary to hold the datasets for each aquifer type
datasets = {}

for name, group in grouped:
    # Assume 'target' is the name of your target variable
    y = group['target']
    # Assume 'features' is a list of your feature names
    X = group[features]
    
    # Split into training and validation datasets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Store the datasets in the datasets dictionary
    datasets[name] = {'X_train': X_train, 'X_val': X_val, 'y_train': y_train, 'y_val': y_val}



In [ ]:
#Need to do 9
aquifer_type_1_datasets = datasets['aquifer_type_1']
X_train = aquifer_type_1_datasets['X_train']
X_val = aquifer_type_1_datasets['X_val']
y_train = aquifer_type_1_datasets['y_train']
y_val = aquifer_type_1_datasets['y_val']


aquifer_type_1_datasets = datasets['aquifer_type_1']
X_train = aquifer_type_1_datasets['X_train']
X_val = aquifer_type_1_datasets['X_val']
y_train = aquifer_type_1_datasets['y_train']
y_val = aquifer_type_1_datasets['y_val']

In [ ]:
from pygam import LinearGAM

# A dictionary to hold the models for each aquifer type
models = {}

for aquifer_type, datasets in datasets.items():
    # Get the training datasets
    X_train = datasets['X_train']
    y_train = datasets['y_train']
    
    # Fit a GAM
    gam = LinearGAM().fit(X_train, y_train)
    
    # Store the model in the models dictionary
    models[aquifer_type] = gam

In [ ]:
from sklearn.metrics import mean_squared_error

aquifer_type_1_model = models['aquifer_type_1']
X_val = datasets['aquifer_type_1']['X_val']
y_val = datasets['aquifer_type_1']['y_val']

y_pred = aquifer_type_1_model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)

print(f'Mean Squared Error for aquifer type 1: {mse}')


In [ ]:
# For each aquifer type
for aquifer_type, model in models.items():
    fig, axs = plt.subplots(1, len(features), figsize=(20, 5))

    # For each feature
    for i, feature in enumerate(features):
        # Plot the learned smooth function for the feature
        model.plot(i, ax=axs[i])

    plt.tight_layout()
    plt.suptitle(f'GAM for {aquifer_type}', fontsize=20)
    plt.show()

In [ ]:
for i, feature in enumerate(features):
    # Get the partial dependency and confidence intervals
    partial_dependence, confidence_intervals = model.partial_dependence(term=i, width=0.95)

    # Plot the partial dependency
    plt.plot(model.terms[i].edge_knots_, partial_dependence, label=feature)

    # Plot the confidence intervals
    plt.fill_between(model.terms[i].edge_knots_, 
                     confidence_intervals[0, :, 0], 
                     confidence_intervals[0, :, 1], 
                     alpha=0.2)

plt.legend()
plt.show()